### DISTINCT vs ROW_NUMBER in Spark SQL
In a perfect world, every record in your table would be unique.
 In reality? Duplicates creep in — sometimes because of bad upstream data, other times due to incorrect joins or multiple data sources merging.

For a data engineer, knowing how to identify and remove duplicates is a critical skill. Spark SQL gives us two powerful ways to do this:

- Using DISTINCT — for simple cases.
- Using ROW_NUMBER() — for advanced control.

Let’s understand both clearly.

### The Setup
Imagine your employees table has accidentally received duplicate entries for some employees due to an ingestion issue.

Employees

![](/Volumes/thedataengineering_00/data/sampledata/sampleimages/employees_duplicates.png)

Clearly, Bob and Diana have duplicate rows. Let’s fix that.

### Method 1: Using DISTINCT
DISTINCT removes duplicates by returning only unique combinations of columns.

### Example 1: Removing complete row duplicates

In [0]:
SELECT DISTINCT *
FROM employees;

This returns only unique rows — any duplicate rows that are exactly identical across all columns will be eliminated.

### Example 2: Unique employees based on name and email

In [0]:
SELECT DISTINCT first_name, last_name, email
FROM employees;

This is useful when you don’t care about other columns like salary or department — you just want unique combinations of these attributes.

Best for: Quick cleanup when duplicates are exact and no tie-breaking logic is needed.Limitation: Doesn’t let you control which duplicate to keep if rows differ slightly (e.g., same employee but different salary).

### Method 2: Using ROW_NUMBER() — When You Need Control
In real-world data, duplicates are often not exact. Maybe one row has an older salary, or one was updated later. In such cases, DISTINCT isn’t enough — you need to decide which record to keep.

That’s where the ROW_NUMBER() window function comes in. It assigns a unique sequential number to each row within a partition, based on ordering.

Example 3: Keeping only the latest record per employee

In [0]:
SELECT *
FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY hire_date DESC) AS row_num
  FROM employees
)
WHERE row_num = 1;

Let’s break it down:

- PARTITION BY emp_id means: treat all rows for the same employee together.
- ORDER BY hire_date DESC means: give priority to the most recent record.
- ROW_NUMBER() assigns 1 to the latest, 2 to the next, and so on.
- The outer query keeps only row_num = 1.

Best for: Datasets where duplicates have variations (different timestamps, salaries, etc.).
 Control: You decide which record wins — the newest, the oldest, or by any other logic.

### Combining Approaches in Practice
Sometimes, you’ll first use DISTINCT to quickly clean the exact duplicates, and then apply ROW_NUMBER() for fine-grained deduplication logic — especially in production pipelines where data comes from multiple sources or ingestion streams.

### Real-World Example for Data Engineers
Imagine you’re building a pipeline that merges daily employee data from multiple HR systems.

- Some systems send duplicate rows for the same employee.
- Others send slightly different versions (maybe salary updated).

In your Spark SQL ETL job, you’d:

Use DISTINCT to remove exact duplicates early.

Use ROW_NUMBER() to retain only the latest version based on last_updated timestamp.

This ensures clean, reliable, and up-to-date data — a core responsibility of any data engineer.

# Closing Thought
Both DISTINCT and ROW_NUMBER() are tools to fight the same problem — data duplication — but they work at different levels of sophistication.

- Use DISTINCT when duplicates are exact and simple.
- Use ROW_NUMBER() when you need to choose which duplicate to keep.

Clean data doesn’t just make queries faster — it makes insights trustworthy.
 For a data engineer, mastering deduplication means mastering data integrity.

In our next session, we’ll move into one of the most powerful parts of SQL analytics: Aggregations and GROUP BY, where you’ll learn how to summarize and measure your data.